# Inventory all data sources in Power BI semantic models

Loops through a known list of semantic models (datasets) in the Power BI / Fabric
Service, reads each model's **Power Query (M) code** via the XMLA / TOM endpoint,
and identifies **every data source** referenced (SQL, Web/API, files, SharePoint,
cloud stores, SaaS connectors, etc.) - not just Airtable/Dropbox, which are still
flagged for convenience.

**Inputs:** a CSV in this Lakehouse with one row per model: a **workspace id**
column and a **dataset id** column (GUIDs).

**Outputs (Delta tables in the attached Lakehouse):**
| table | contents |
|---|---|
| `migration_source_detail` | one row per (model, object, connector, detail) with full M code |
| `migration_source_summary` | distinct data sources and which models use them |
| `migration_source_by_category` | rollup of source categories |
| `migration_scan_errors` | models that could not be scanned, with the error |

**Requirements**
* A **default Lakehouse** attached to this notebook.
* **XMLA endpoint = Read** on the capacity, and at least *Read/Build* on each model.
* `semantic-link-labs` installed (next cell).

In [ ]:
# Install the TOM helper. Safe to re-run; restart not required within the session.
%pip install -q semantic-link-labs

## Parameters
Tagged **parameters** so it can be overridden from a pipeline. Set `top_n` to a
small number to test on a subset, or `None` to process every model.

In [ ]:
# ---- CSV input ------------------------------------------------------------
csv_path        = "Files/pbi_audit/prod_reports_hitlist.csv"  # path in the attached Lakehouse
workspace_id_col = "WorkspaceId"                # column holding the workspace GUID
dataset_id_col   = "SemanticModelId"            # column holding the dataset  GUID

# ---- Testing knob ---------------------------------------------------------
top_n = None         # int -> only process the first N models; None -> all of them

# ---- Output ---------------------------------------------------------------
output_prefix = "migration_"   # Delta tables get this prefix
write_tables  = True           # set False to only display, not persist

In [ ]:
import re
import datetime
import pandas as pd
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model

# Any "Namespace.Function(" call in the M, e.g. Sql.Database( , Web.Contents(
CALL_RE   = re.compile(r'\b([A-Z][A-Za-z0-9]*\.[A-Za-z0-9]+)\s*\(')
QUOTED_RE = re.compile(r'"([^"]*)"')

# Known Power Query data-source / connector functions -> friendly category.
DATA_SOURCE_FUNCS = {
    # relational databases
    "Sql.Database": "SQL Server", "Sql.Databases": "SQL Server",
    "PostgreSQL.Database": "PostgreSQL", "PostgreSQL.Databases": "PostgreSQL",
    "MySQL.Database": "MySQL", "MySQL.Databases": "MySQL",
    "Oracle.Database": "Oracle",
    "Db2.Database": "IBM Db2",
    "Teradata.Database": "Teradata",
    "Sybase.Database": "Sybase",
    "Informix.Database": "Informix",
    "Vertica.Database": "Vertica",
    "Impala.Database": "Impala",
    "Snowflake.Databases": "Snowflake", "Snowflake.Database": "Snowflake",
    "AmazonRedshift.Database": "Amazon Redshift",
    "GoogleBigQuery.Database": "Google BigQuery",
    "Databricks.Catalogs": "Databricks", "Databricks.Query": "Databricks",
    "SapHana.Database": "SAP HANA",
    "SapBusinessWarehouse.Cubes": "SAP BW",
    "AnalysisServices.Database": "Analysis Services",
    "AnalysisServices.Databases": "Analysis Services",
    "Kusto.Contents": "Azure Data Explorer",
    "AzureDataExplorer.Contents": "Azure Data Explorer",
    # files
    "File.Contents": "File", "Folder.Contents": "Folder", "Folder.Files": "Folder",
    "Excel.Workbook": "Excel", "Csv.Document": "CSV / Text", "Json.Document": "JSON",
    "Xml.Tables": "XML", "Xml.Document": "XML", "Pdf.Tables": "PDF",
    "Parquet.Document": "Parquet",
    # web / api / generic providers
    "Web.Contents": "Web / API", "Web.Page": "Web", "Web.BrowserContents": "Web",
    "OData.Feed": "OData", "Odbc.DataSource": "ODBC", "Odbc.Query": "ODBC",
    "OleDb.DataSource": "OLE DB",
    # cloud storage
    "AzureStorage.Blobs": "Azure Blob Storage",
    "AzureStorage.Tables": "Azure Table Storage",
    "AzureStorage.DataLake": "Azure Data Lake",
    "AzureStorage.DataLakeContents": "Azure Data Lake",
    "DataLake.Contents": "Azure Data Lake",
    # sharepoint / m365
    "SharePoint.Files": "SharePoint", "SharePoint.Contents": "SharePoint",
    "SharePoint.Tables": "SharePoint", "SharePointList.Tables": "SharePoint List",
    # saas
    "Salesforce.Data": "Salesforce", "Salesforce.Reports": "Salesforce",
    "GoogleAnalytics.Accounts": "Google Analytics",
    "Dynamics365.Contents": "Dynamics 365",
    "CommonDataService.Database": "Dataverse", "Cds.Contents": "Dataverse",
    "PowerPlatform.Dataflows": "Power Platform Dataflows",
    "PowerBI.Datasets": "Power BI dataset",
    # fabric
    "Lakehouse.Contents": "Fabric Lakehouse", "Fabric.Warehouse": "Fabric Warehouse",
}

# Functions not in the map whose name ends with one of these are treated as a
# likely (but unrecognised) connector and tagged "Other (<namespace>)".
SOURCE_SUFFIXES = (".Database", ".Databases", ".Contents", ".Document", ".Workbook",
                   ".Feed", ".DataSource", ".Files", ".Tables", ".Cubes",
                   ".Catalogs", ".Query", ".Accounts", ".Blobs", ".Data", ".Reports")

def _find_sources(m_code):
    # Return list of (connector_function, category, primary_detail) found in the M.
    out = []
    for m in CALL_RE.finditer(m_code):
        fname = m.group(1)
        category = DATA_SOURCE_FUNCS.get(fname)
        if category is None:
            if any(fname.endswith(s) for s in SOURCE_SUFFIXES):
                category = "Other (" + fname.split(".")[0] + ")"
            else:
                continue
        # First couple of string literals after "(" are usually server/db/url/path.
        tail = m_code[m.end(): m.end() + 200]
        args = [a for a in QUOTED_RE.findall(tail) if a][:2]
        out.append((fname, category, " , ".join(args)))
    return out

def _classify(ws, ds, obj_type, obj_name, m_code):
    # One row per distinct (connector, detail) data source in this object's M.
    m_code = m_code or ""
    low = m_code.lower()
    flags = ",".join(k for k in ("airtable", "dropbox") if k in low)
    rows, seen = [], set()
    for fname, category, detail in _find_sources(m_code):
        key = (fname, detail)
        if key in seen:
            continue
        seen.add(key)
        rows.append({
            "workspace_id": ws, "dataset_id": ds,
            "object_type": obj_type, "object_name": obj_name,
            "connector": fname, "source_category": category,
            "source_detail": detail, "keyword_flags": flags,
            "m_code": m_code,
        })
    return rows

def scan_model(ws, ds):
    # Connect to one semantic model and pull M from named expressions + partitions.
    rows = []
    with connect_semantic_model(dataset=ds, workspace=ws, readonly=True) as tom:
        for e in tom.model.Expressions:
            rows += _classify(ws, ds, "Expression", e.Name, str(e.Expression))
        for t in tom.model.Tables:
            for p in t.Partitions:
                try:
                    expr = p.Source.Expression
                except Exception:
                    expr = None      # calculated / entity / DirectQuery partitions
                if expr:
                    rows += _classify(ws, ds, "Partition", t.Name + "." + p.Name, str(expr))
    return rows

In [ ]:
# Read the model list from the Lakehouse and apply the top_n test limit.
models_pdf = (
    spark.read.option("header", True).csv(csv_path)
    .select(workspace_id_col, dataset_id_col)
    .toPandas()
    .rename(columns={workspace_id_col: "workspace_id", dataset_id_col: "dataset_id"})
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

if top_n is not None:
    models_pdf = models_pdf.head(int(top_n))

print(f"{len(models_pdf)} model(s) to scan"
      + (f" (limited to top {top_n})" if top_n is not None else ""))
display(models_pdf)

In [ ]:
# Main loop: scan each model, collecting detail rows and any per-model errors.
detail_rows, error_rows = [], []
total = len(models_pdf)

for i, row in models_pdf.iterrows():
    ws, ds = row["workspace_id"], row["dataset_id"]
    try:
        found = scan_model(ws, ds)
        detail_rows.extend(found)
        n_conn = len({r["connector"] for r in found})
        print(f"[{i + 1}/{total}] {ds}: {len(found)} source ref(s), {n_conn} connector type(s)")
    except Exception as ex:
        error_rows.append({"workspace_id": ws, "dataset_id": ds, "error": f"{type(ex).__name__}: {ex}"})
        print(f"[{i + 1}/{total}] {ds}: ERROR - {type(ex).__name__}: {ex}")

detail_pdf = pd.DataFrame(detail_rows, columns=[
    "workspace_id", "dataset_id", "object_type", "object_name",
    "connector", "source_category", "source_detail", "keyword_flags", "m_code"])
errors_pdf = pd.DataFrame(error_rows, columns=["workspace_id", "dataset_id", "error"])

print(f"\nDone. {len(detail_pdf)} source ref(s) across "
      f"{detail_pdf['dataset_id'].nunique() if len(detail_pdf) else 0} model(s); "
      f"{len(errors_pdf)} error(s).")
display(detail_pdf)

In [ ]:
# Distinct data sources + a category rollup across all scanned models.
if len(detail_pdf):
    summary_pdf = (
        detail_pdf
        .groupby(["source_category", "connector", "source_detail"])
        .agg(model_count=("dataset_id", "nunique"),
             ref_count=("object_name", "count"),
             dataset_ids=("dataset_id", lambda s: " | ".join(sorted(set(s)))))
        .reset_index()
        .sort_values(["source_category", "model_count"], ascending=[True, False])
    )
    category_pdf = (
        detail_pdf.groupby("source_category")
        .agg(model_count=("dataset_id", "nunique"), ref_count=("object_name", "count"))
        .reset_index().sort_values("ref_count", ascending=False)
    )
else:
    summary_pdf = pd.DataFrame(columns=["source_category", "connector", "source_detail",
                                        "model_count", "ref_count", "dataset_ids"])
    category_pdf = pd.DataFrame(columns=["source_category", "model_count", "ref_count"])

print(f"{len(summary_pdf)} distinct data source(s) across {len(category_pdf)} category type(s).")
display(category_pdf)
display(summary_pdf)

In [ ]:
# Persist results as Delta tables in the attached Lakehouse.
def _save(pdf, name):
    if pdf.empty:
        print(f"  (skipped {name}: no rows)")
        return
    (spark.createDataFrame(pdf)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(name))
    print(f"  wrote {name} ({len(pdf)} rows)")

if write_tables:
    print("Writing tables:")
    _save(detail_pdf,   output_prefix + "source_detail")
    _save(summary_pdf,  output_prefix + "source_summary")
    _save(category_pdf, output_prefix + "source_by_category")
    _save(errors_pdf,   output_prefix + "scan_errors")
else:
    print("write_tables=False - nothing persisted.")

## Notes & troubleshooting

* **Coverage** - the curated `DATA_SOURCE_FUNCS` list maps the common Power Query
  connectors to friendly names; any other `Namespace.Function(` that looks like a
  connector is still captured as `Other (<namespace>)`, so review those and add
  them to the map if needed. The full `m_code` column is always retained for
  verification.
* **`source_detail`** holds the first one or two string literals after the
  connector call (typically server / database / url / file path). When the source
  is built from parameters rather than literals it may be blank - check `m_code`.
* **Access / XMLA errors** land in `migration_scan_errors` rather than stopping the
  run. Grant Read/Build on the model and enable *XMLA endpoint = Read*.
* **Performance** - sequential, one XMLA session per model; use `top_n` to validate
  on a few first. Tables are overwritten each run; change `output_prefix` to keep
  separate outputs.